## SRP099173 / GSE94871

**paper:** [PMID: 28296881](https://pmc.ncbi.nlm.nih.gov/articles/PMC5352168/) - Transcriptomic responses to environmental temperature by turtles with temperature-dependent and genotypic sex determination assessed by RNAseq inform the genetic architecture of embryonic gonadal development, 2017

**date, curator:** 2026-04-29, Sara Carsanaro

**notes**
* adrenal-kidney-gonad complex is not in uberon right now - to make a request
    * annotating as genitourinary system because the other option is like organ system or intermediate mesoderm or something super vague
* dev stages - see [An enhanced developmental staging table for the painted turtle, Chrysemys picta (Testudines: Emydidae)](https://onlinelibrary.wiley.com/doi/10.1002/jmor.20226)
    * 9 - 20 paired somites, closed neuropores - organogenesis stage
    * 12 - many (over 30 pairs) somites, 5th pharyngeal slit is visible - organogenesis stage
    * 15 - cervical sinus closed, carapacial ridge extended the entire flank of the embryo - this is still organogenesis stage i think but really the end of it
    * 19 - lower eyelid growing, upper eyelid present, digits extend beyond webbing, increased pigmentation - late embryonic stage
    * 22 - stage 23 is hatching so this is the penultimate stage of development. eyelids extend over the iris, intestinal loop is withdrawn, body is thoroughly pigmented - late embryonic stage

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,gonad,UBERON:0000991,gonad,perfect match
2,adrenal-kidney-gonad complex,UBERON:0004122,genitourinary system,other
4,trunks,UBERON:0002100,trunk,perfect match
10,gonad,UBERON:0000992,ovary,perfect match
15,gonad,UBERON:0000473,testis,perfect match


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,ASP stage 22,UBERON:0007220,late embryonic stage,other
1,ASP stage 19,UBERON:0007220,late embryonic stage,other
2,ASP stage 15,UBERON:0000111,organogenesis stage,other
3,ASP stage 12,UBERON:0000111,organogenesis stage,other
4,ASP stage 9,UBERON:0000111,organogenesis stage,other
10,CPI stage 22,UBERON:0007220,late embryonic stage,other
11,CPI stage 19,UBERON:0007220,late embryonic stage,other
12,CPI stage 15,UBERON:0000111,organogenesis stage,other
13,CPI stage 12,UBERON:0000111,organogenesis stage,other
14,CPI stage 9,UBERON:0000111,organogenesis stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP099173"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
29-Apr-2026 14:01:46 DEBUG utils - Directory ./ already exists. Skipping.
29-Apr-2026 14:01:46 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE94nnn/GSE94871/soft/GSE94871_family.soft.gz to ./GSE94871_family.soft.gz
100%|██████████████████████████████████████| 2.62k/2.62k [00:00<00:00, 5.73kB/s]
29-Apr-2026 14:01:47 DEBUG downloader - Size validation passed
29-Apr-2026 14:01:47 DEBUG downloader - Moving /var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/tmpudg_0ckn to /Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/SRP099173/GSE94871_family.soft.gz
29-Apr-2026 14

### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2549101,SRP099173,Illumina HiSeq 2000,SRS1967563,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,not determined,,,55534,,,,,,ASP_31_22,"SAMN06298522,GSM2486800",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_22,,,,ASP stage 22,,TRANSCRIPTOMIC,RANDOM
1,SRX2549102,SRP099173,Illumina HiSeq 2000,SRS1967564,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 19,perfect match,not documented,other,not determined,,,55534,,,,,,ASP_31_19,"SAMN06298521,GSM2486799",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_19,,,,ASP stage 19,,TRANSCRIPTOMIC,RANDOM
2,SRX2549103,SRP099173,Illumina HiSeq 2000,SRS1967565,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 15,other,not documented,other,not determined,,,55534,,,,,,ASP_31_15,"SAMN06298520,GSM2486798",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_15,,,,ASP stage 15,,TRANSCRIPTOMIC,RANDOM
3,SRX2549104,SRP099173,Illumina HiSeq 2000,SRS1967566,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 12,other,not documented,other,not determined,,,55534,,,,,,ASP_31_12,"SAMN06298519,GSM2486797",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_12,,,,ASP stage 12,,TRANSCRIPTOMIC,RANDOM
4,SRX2549105,SRP099173,Illumina HiSeq 2000,SRS1967567,UBERON:0002100,trunk,UBERON:0000111,organogenesis stage,,trunks,ASP stage 9,perfect match,not documented,other,not determined,,,55534,,,,,,ASP_31_9,"SAMN06298518,GSM2486796",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_9,,,,ASP stage 9,,TRANSCRIPTOMIC,RANDOM
5,SRX2549106,SRP099173,Illumina HiSeq 2000,SRS1967568,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,pooled male and female,,,55534,,,,,,ASP_26_22,"SAMN06298517,GSM2486795",,,,,,,,26 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) 

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['adrenal-kidney-gonad complex' 'gonad' 'trunks']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['ASP stage 12' 'ASP stage 15' 'ASP stage 19' 'ASP stage 22' 'ASP stage 9'
 'CPI stage 12' 'CPI stage 15' 'CPI stage 19' 'CPI stage 22' 'CPI stage 9']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [7]:

library.loc[library["sex"] == "pooled male and female", "sex"] = "mixed"
library.loc[library["sex"] == "not determined", "sex"] = "NA"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2549101,SRP099173,Illumina HiSeq 2000,SRS1967563,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,NA,,,55534,,,,,,ASP_31_22,"SAMN06298522,GSM2486800",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_22,,,,ASP stage 22,,TRANSCRIPTOMIC,RANDOM
1,SRX2549102,SRP099173,Illumina HiSeq 2000,SRS1967564,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 19,perfect match,not documented,other,NA,,,55534,,,,,,ASP_31_19,"SAMN06298521,GSM2486799",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_19,,,,ASP stage 19,,TRANSCRIPTOMIC,RANDOM
2,SRX2549103,SRP099173,Illumina HiSeq 2000,SRS1967565,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 15,other,not documented,other,NA,,,55534,,,,,,ASP_31_15,"SAMN06298520,GSM2486798",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_15,,,,ASP stage 15,,TRANSCRIPTOMIC,RANDOM
3,SRX2549104,SRP099173,Illumina HiSeq 2000,SRS1967566,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 12,other,not documented,other,NA,,,55534,,,,,,ASP_31_12,"SAMN06298519,GSM2486797",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_12,,,,ASP stage 12,,TRANSCRIPTOMIC,RANDOM
4,SRX2549105,SRP099173,Illumina HiSeq 2000,SRS1967567,UBERON:0002100,trunk,UBERON:0000111,organogenesis stage,,trunks,ASP stage 9,perfect match,not documented,other,NA,,,55534,,,,,,ASP_31_9,"SAMN06298518,GSM2486796",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_9,,,,ASP stage 9,,TRANSCRIPTOMIC,RANDOM
5,SRX2549106,SRP099173,Illumina HiSeq 2000,SRS1967568,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,mixed,,,55534,,,,,,ASP_26_22,"SAMN06298517,GSM2486795",,,,,,,,26 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzu

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2549101,SRP099173,Illumina HiSeq 2000,SRS1967563,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_22,"SAMN06298522,GSM2486800",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_22,,,,ASP stage 22,,TRANSCRIPTOMIC,RANDOM
1,SRX2549102,SRP099173,Illumina HiSeq 2000,SRS1967564,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 19,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_19,"SAMN06298521,GSM2486799",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_19,,,,ASP stage 19,,TRANSCRIPTOMIC,RANDOM
2,SRX2549103,SRP099173,Illumina HiSeq 2000,SRS1967565,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 15,other,not documented,other,NA,,,55534,,,polyA,,,ASP_31_15,"SAMN06298520,GSM2486798",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_15,,,,ASP stage 15,,TRANSCRIPTOMIC,RANDOM
3,SRX2549104,SRP099173,Illumina HiSeq 2000,SRS1967566,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 12,other,not documented,other,NA,,,55534,,,polyA,,,ASP_31_12,"SAMN06298519,GSM2486797",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_12,,,,ASP stage 12,,TRANSCRIPTOMIC,RANDOM
4,SRX2549105,SRP099173,Illumina HiSeq 2000,SRS1967567,UBERON:0002100,trunk,UBERON:0000111,organogenesis stage,,trunks,ASP stage 9,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_9,"SAMN06298518,GSM2486796",,,,,,,,31 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_9,,,,ASP stage 9,,TRANSCRIPTOMIC,RANDOM
5,SRX2549106,SRP099173,Illumina HiSeq 2000,SRS1967568,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,mixed,,,55534,,,polyA,,,ASP_26_22,"SAMN06298517,GSM2486795",,,,,,,,26 C,,,29/04/2026,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stage

#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-30'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2549101,SRP099173,Illumina HiSeq 2000,SRS1967563,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_22,"SAMN06298522,GSM2486800",,,,,,,,31 C,,SAC,2026-04-30,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_22,,,,ASP stage 22,,TRANSCRIPTOMIC,RANDOM
1,SRX2549102,SRP099173,Illumina HiSeq 2000,SRS1967564,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 19,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_19,"SAMN06298521,GSM2486799",,,,,,,,31 C,,SAC,2026-04-30,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_19,,,,ASP stage 19,,TRANSCRIPTOMIC,RANDOM
2,SRX2549103,SRP099173,Illumina HiSeq 2000,SRS1967565,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 15,other,not documented,other,NA,,,55534,,,polyA,,,ASP_31_15,"SAMN06298520,GSM2486798",,,,,,,,31 C,,SAC,2026-04-30,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_15,,,,ASP stage 15,,TRANSCRIPTOMIC,RANDOM
3,SRX2549104,SRP099173,Illumina HiSeq 2000,SRS1967566,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 12,other,not documented,other,NA,,,55534,,,polyA,,,ASP_31_12,"SAMN06298519,GSM2486797",,,,,,,,31 C,,SAC,2026-04-30,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_12,,,,ASP stage 12,,TRANSCRIPTOMIC,RANDOM
4,SRX2549105,SRP099173,Illumina HiSeq 2000,SRS1967567,UBERON:0002100,trunk,UBERON:0000111,organogenesis stage,,trunks,ASP stage 9,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_9,"SAMN06298518,GSM2486796",,,,,,,,31 C,,SAC,2026-04-30,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and gonads alone (stages 19, 22). RNeasy Kits(Valenzuela,2008a). RNA was quanti-fied with a NanoDropVRND-1000 Spectrophotometer,and its quality assessed by the presence of ribosomal bands in agarose gels",,ASP_31_9,,,,ASP stage 9,,TRANSCRIPTOMIC,RANDOM
5,SRX2549106,SRP099173,Illumina HiSeq 2000,SRS1967568,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,mixed,,,55534,,,polyA,,,ASP_26_22,"SAMN06298517,GSM2486795",,,,,,,,26 C,,SAC,2026-04-30,"Total RNA was extracted from trunks (stage-9), adrenal-kidney-gonadal (AKG) complex (stages 12, 15) and g

#### comments

In [11]:
library.loc[:,'comment'] = 'PMID: 28296881'

#### save complete file with correct columns

In [12]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX2549101,SRP099173,Illumina HiSeq 2000,SRS1967563,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_22,"SAMN06298522,GSM2486800",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30
1,SRX2549102,SRP099173,Illumina HiSeq 2000,SRS1967564,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 19,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_19,"SAMN06298521,GSM2486799",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30
2,SRX2549103,SRP099173,Illumina HiSeq 2000,SRS1967565,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 15,other,not documented,other,NA,,,55534,,,polyA,,,ASP_31_15,"SAMN06298520,GSM2486798",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30
3,SRX2549104,SRP099173,Illumina HiSeq 2000,SRS1967566,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 12,other,not documented,other,NA,,,55534,,,polyA,,,ASP_31_12,"SAMN06298519,GSM2486797",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30
4,SRX2549105,SRP099173,Illumina HiSeq 2000,SRS1967567,UBERON:0002100,trunk,UBERON:0000111,organogenesis stage,,trunks,ASP stage 9,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_9,"SAMN06298518,GSM2486796",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30
5,SRX2549106,SRP099173,Illumina HiSeq 2000,SRS1967568,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,mixed,,,55534,,,polyA,,,ASP_26_22,"SAMN06298517,GSM2486795",,,,,,,PMID: 28296881,26 C,,SAC,2026-04-30
6,SRX2549107,SRP099173,Illumina HiSeq 2000,SRS1967570,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 19,perfect match,not documented,other,mixed,,,55534,,,polyA,,,ASP_26_19,"SAMN06298516,GSM2486794",,,,,,,PMID: 28296881,26 C,,SAC,2026-04-30
7,SRX2549108,SRP099173,Illumina HiSeq 2000,SRS1967569,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 15,other,not documented,other,mixed,,,55534,,,polyA,,,ASP_26_15,"SAMN06298515,GSM2486793",,,,,,,PMID: 28296881,26 C,,SAC,2026-04-30
8,SRX2549109,SRP099173,Illumina HiSeq 2000,SRS1967572,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 12,other,not documented,other,mixed,,,55534,,,polyA,,,ASP_26_12,"SAMN06298514,GSM2486792",,,,,,,PMID: 28296881,26 C,,SAC,2026-04-30
9,SRX2549110,SRP099173,Illumina HiSeq 2000,SRS1967571,UBERON:0002100,trunk,UBERON:0000111,organogenesis stage,,trunks,ASP stage 9,perfect match,not documented,other,mixed,,,55534,,,polyA,,,ASP_26_9,"SAMN06298513,GSM2486791",,,,,,,PMID: 28296881,26 C,,SAC,2026-04-30


### experiment annotations

In [13]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP099173,"Time series embryonic gonadal transcriptome of the painted turtle Chrysemys picta prior-to, at the onset of, during, and end of the thermosensitive period. Raw sequence reads",Study of embryonic gonadal transcriptome across 5 developmental stages of turtles with genotypic and temperature-dependent sex determination.,SRA,,,,,,GSE94871,PRJNA371383,28296881,,10.1371/journal.pone.0172044,,


#### experiment and protocol details

In [14]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

20

In [15]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP099173,"Time series embryonic gonadal transcriptome of the painted turtle Chrysemys picta prior-to, at the onset of, during, and end of the thermosensitive period. Raw sequence reads",Study of embryonic gonadal transcriptome across 5 developmental stages of turtles with genotypic and temperature-dependent sex determination.,SRA,total,Bgee 1K,20,,,GSE94871,PRJNA371383,28296881,,10.1371/journal.pone.0172044,,


#### paper and xrefs

In [16]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
##experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5352168/'
#experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP099173,"Time series embryonic gonadal transcriptome of the painted turtle Chrysemys picta prior-to, at the onset of, during, and end of the thermosensitive period. Raw sequence reads",Study of embryonic gonadal transcriptome across 5 developmental stages of turtles with genotypic and temperature-dependent sex determination.,SRA,total,Bgee 1K,20,,,GSE94871,PRJNA371383,28296881,https://pmc.ncbi.nlm.nih.gov/articles/PMC5352168/,10.1371/journal.pone.0172044,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [17]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [18]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [19]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [22]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [23]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
62676,SRX1874777,SRP077056,Illumina HiSeq 2500,SRS1524454,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 44 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,10,CP22,SAMN05220590,44,embryonic day,,,,,PMID: 30094964,Stage 22,,SAC,2026-04-29
62677,SRX1874776,SRP077056,Illumina HiSeq 2500,SRS1524454,UBERON:0001421,pectoral girdle region,UBERON:0007220,late embryonic stage,,Dorsal scapula,Day 44 embryo,other,not documented,missing child term,NA,,,8479,TruSeq RNA Sample Preparation Kit,full_length,polyA,,10,CP22,SAMN05220590,44,embryonic day,,,,,PMID: 30094964,Stage 22,,SAC,2026-04-29
62678,SRX2549101,SRP099173,Illumina HiSeq 2000,SRS1967563,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 22,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_22,"SAMN06298522,GSM2486800",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30
62679,SRX2549102,SRP099173,Illumina HiSeq 2000,SRS1967564,UBERON:0000991,gonad,UBERON:0007220,late embryonic stage,,gonad,ASP stage 19,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_19,"SAMN06298521,GSM2486799",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30
62680,SRX2549103,SRP099173,Illumina HiSeq 2000,SRS1967565,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 15,other,not documented,other,NA,,,55534,,,polyA,,,ASP_31_15,"SAMN06298520,GSM2486798",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30
62681,SRX2549104,SRP099173,Illumina HiSeq 2000,SRS1967566,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,ASP stage 12,other,not documented,other,NA,,,55534,,,polyA,,,ASP_31_12,"SAMN06298519,GSM2486797",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30
62682,SRX2549105,SRP099173,Illumina HiSeq 2000,SRS1967567,UBERON:0002100,trunk,UBERON:0000111,organogenesis stage,,trunks,ASP stage 9,perfect match,not documented,other,NA,,,55534,,,polyA,,,ASP_31_9,"SAMN06298518,GSM2486796",,,,,,,PMID: 28296881,31 C,,SAC,2026-04-30


In [24]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1210,SRP586826,zebra finch and budgerigar Raw sequence reads,The 16S rRNA gene and hepatic transcriptomic s...,SRA,partial,Bgee 1K,37,,,,PRJNA1094658,,,,,"no PMID, newer data, to revisit and look for p..."
1211,SRP077056,Gene network variation and alternative paths t...,Diversification of the turtle's shell comprise...,SRA,total,Bgee 1K,30,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA323738,30094964,https://onlinelibrary.wiley.com/doi/epdf/10.11...,10.1111/ede.12264,,
1212,SRP099173,Time series embryonic gonadal transcriptome of...,Study of embryonic gonadal transcriptome acros...,SRA,total,Bgee 1K,20,,,GSE94871,PRJNA371383,28296881,https://pmc.ncbi.nlm.nih.gov/articles/PMC5352168/,10.1371/journal.pone.0172044,,


### add annotations to git

In [25]:
! git pull

Already up to date.


In [26]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [27]:
! git add $git_experiment_path $git_library_path

In [28]:
! git commit -m $commit_message_exp

[develop bfe6e01] adding annotated bulk experiment SRP099173
 2 files changed, 21 insertions(+)


In [29]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.65 KiB | 679.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   c554401..bfe6e01  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push